In [ ]:
import pathlib
import re

import numpy as np
import pandas as pd
import pyarrow as pa

from pyarrow import parquet as pq

from MDAnalysis.analysis.rms import rmsd

In [ ]:
cwd = pathlib.Path.cwd()
quantum_green = pathlib.Path("/home/shared/projects/quantum_green")
paquets = cwd / "parquets"

`ts_data`: Contains the consolidated Transition State data from the different databases.

In [ ]:
ts_data = pd.read_pickle(quantum_green / "ts" / "quantum_green_ts_data_24july2c.pkl")

In [ ]:
ts_data.columns.tolist()

In [ ]:
ts_data["batch_label"].value_counts()

In [ ]:
for label in ["ts_nho_round1", "ocoo_g4mp2", "oo_g4mp2", "ocoo_grp", "oo_grp"]:
    dft_paths = ts_data[["ts_dft_log_filename_id", "ts_dft_log_path"]][ts_data["batch_label"] == label]
    dft_paths.to_csv(cwd / f"{label}.csv", index=False)

### Mapping DFT to DFT

`dft_sources`: Contains the paths of the log files for the DFT calculations for the transition states.

In [ ]:
pq.read_schema(paquets / "dft_ts_dft_v9.parquet")

In [ ]:

dft_sources = pa.concat_tables(
    [
        pq.read_table(
            paquets / f"dft_ts_dft_{case}.parquet",
            columns=["source", "route_section"],
        )
        for case in ["v9", "nho_oho_v9"]
    ]
).to_pandas()
dft_sources["source"] = dft_sources["source"].apply(
    lambda x: x.replace("/data1/", "/home/gridsan/")
)
dft_sources["dft_index"] = dft_sources.index
dft_sources.head()

`ts_data_sources`: Maps the `ts_data` index to the source of the DFT calculation.

In [ ]:
dft_dtf_mapping = dict(zip(dft_sources["source"], dft_sources.index))
ts_data["dft_index"] = ts_data["ts_dft_log_path"].apply(
    lambda x: dft_dtf_mapping.get(x, -1)
)
(ts_data["dft_index"] == -1).sum()

### Mapping DFT to Semi-empirical

`semi_sources`: Contains the paths of the log files for the semi-empirical calculations for the transition states.

In [ ]:
semi_sources = pa.concat_tables(
    [
        pq.read_table(
            paquets / f"semiempirical_ts_semiempirical_{case}.parquet",
            columns=["source", "route_section", "gibbs", "e0_zpe", "e0_h"],
        )
        for case in ["v9", "nho_oho_1_v9", "nho_oho_2_v9", "nho_oho_3_v9"]
    ]
).to_pandas()
semi_sources["source"] = semi_sources["source"].apply(
    lambda x: x.replace("/data1/", "/home/gridsan/")
)
len(semi_sources)

In [ ]:
INCLUDE_ROUND_0 = True
INCLUDE_ROUND_1 = True
INCLUDE_ROUND_2 = False

DISTINGUISH_ROUNDS = False

PRIORITIZE_LABELS_VS_ENERGIES = True

if PRIORITIZE_LABELS_VS_ENERGIES:
    priority_order = ["batch_label", "id", "theory_level", "gibbs", "e0_zpe", "e0_h"]
else:
    priority_order = ["gibbs", "e0_zpe", "e0_h", "batch_label", "id", "theory_level"]

In [ ]:
HABS_PREFIX = "/home/gridsan/groups/RMG/Projects/Habs/data/ts/semi"
CO2_CAPTURE_PREFIX = "/home/gridsan/groups/co2_capture/ts_guess_generation/nho_oho_ts_guess_generation_20220524/generated_ts_guesses"

def get_batch_label(source):
    if source.startswith(HABS_PREFIX) and "co2_first_round_semi_log" not in source:
        return source.split("/")[10]
    if INCLUDE_ROUND_0 and source.startswith(f"{CO2_CAPTURE_PREFIX}/round_0"):
        return f"ts_nho_round{0 if DISTINGUISH_ROUNDS else 1}"
    if INCLUDE_ROUND_1 and source.startswith(f"{CO2_CAPTURE_PREFIX}/round_1"):
        return "ts_nho_round1"
    if INCLUDE_ROUND_2 and source.startswith(f"{CO2_CAPTURE_PREFIX}/round_2"):
        return f"ts_nho_round{2 if DISTINGUISH_ROUNDS else 1}"
    return "Unknown"


def get_id(source):
    if source.startswith(HABS_PREFIX):
        return int(source.split("/")[-1].split("_")[1])
    if source.startswith(CO2_CAPTURE_PREFIX):
        return int(source.split("/")[-2].split("_")[1])
    return -1


def get_theory_level(route_section):
    if route_section is None:
        return "Unknown"
    if route_section.endswith("am1"):
        return "am1"
    if route_section.endswith("pm7"):
        return "pm7"
    if route_section.endswith("xtb_gaussian.pl --gfn 2 -P\""):
        return "xtb"
    return "Unknown"

semi_sources["batch_label"] = semi_sources["source"].apply(get_batch_label)
semi_sources["id"] = semi_sources["source"].apply(get_id)
semi_sources["theory_level"] = semi_sources["route_section"].apply(get_theory_level)
semi_sources["semiempirical_index"] = semi_sources.index

semi_sources

The following cell filters the `semi_sources` DataFrame to retain only valid entries for downstream analysis.
It removes rows where the batch label, id, or theory level are unknown or invalid, and drops rows with missing
values in the "gibbs", "e0_zpe", or "e0_h" columns. The filtered DataFrame is then sorted by batch label, id,
theory level, and the three energy columns. The resulting DataFrame, `filtered_semi_sources`, is displayed.

In [ ]:
filtered_semi_sources = semi_sources[
    (semi_sources["batch_label"] != "Unknown")
    & (semi_sources["id"] != -1)
    & (semi_sources["theory_level"] != "Unknown")
].dropna(
    subset=["gibbs", "e0_zpe", "e0_h"]
).sort_values(by=priority_order)

filtered_semi_sources

In [ ]:
duplicated_semi_sources = filtered_semi_sources[
    filtered_semi_sources.duplicated(subset=["batch_label", "id", "theory_level"], keep=False)
]
len(duplicated_semi_sources)

In [ ]:
if DISTINGUISH_ROUNDS:
    duplicated_semi_sources.to_csv(cwd / "duplicated_semi_sources.csv", index=False)

Duplicates (with respect to batch label, id, and theory level)
are removed, keeping only the first occurrence

In [ ]:
filtered_semi_sources.drop_duplicates(
    subset=["batch_label", "id", "theory_level"], keep="first", inplace=True
)
filtered_semi_sources

In [ ]:
mapping = dict(
    zip(
        (
            filtered_semi_sources["batch_label"]
            + "+"
            + filtered_semi_sources["id"].astype(str)
            + "+"
            + filtered_semi_sources["theory_level"]
        ),
        filtered_semi_sources["semiempirical_index"],
    )
)
mapping

In [ ]:
for theory_level in ["am1", "pm7", "xtb"]:
    ts_data[f"{theory_level}_index"] = (
        ts_data["batch_label"] + "+" + ts_data["ts_dft_log_filename_id"].astype(str) + f"+{theory_level}"
    ).apply(lambda x: mapping.get(x, -1))

In [ ]:
ts_data[
    (ts_data["am1_index"] == -1)
    | (ts_data["pm7_index"] == -1)
    | (ts_data["xtb_index"] == -1)
]

In [ ]:
matching_df = ts_data[["dft_table_id", "dft_index", "am1_index", "pm7_index", "xtb_index"]]
matching_df

In [ ]:
matching_df.to_csv("quantum_green_ts_data_match.csv", index=False)